# 10 — Local LLM with Ollama

Run LangChain entirely locally — no API keys required.

**Prerequisites:** Install [Ollama](https://ollama.com) and run `ollama pull llama3.2`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama, OllamaEmbeddings

MODEL_NAME = 'llama3.2'

## Example 1: Basic Local Chat

In [ ]:
llm = ChatOllama(model=MODEL_NAME, temperature=0)
response = llm.invoke("What are three benefits of open-source software? Be brief.")
print(response.content)

## Example 2: Local Chain

In [ ]:
prompt = ChatPromptTemplate.from_template("You are a {role}. Answer this question in 2-3 sentences:\n{question}")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"role": "friendly science teacher", "question": "Why do leaves change color in autumn?"})
print(result)

## Example 3: Full Local RAG

In [ ]:
from langchain_chroma import Chroma

docs = [
    Document(page_content="Ollama makes it easy to run large language models locally. It supports models like Llama, Mistral, and Gemma."),
    Document(page_content="Local LLMs offer privacy advantages since data never leaves your machine. They work offline and have predictable costs."),
    Document(page_content="Popular local models include Llama 3.2 (Meta), Mistral (Mistral AI), and Gemma (Google). These range from 1B to 70B+ parameters."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = splitter.split_documents(docs)

embeddings = OllamaEmbeddings(model=MODEL_NAME)
vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

prompt = ChatPromptTemplate.from_template("Answer based on this context only:\n{context}\n\nQuestion: {question}\nAnswer:")

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "What are the benefits of running LLMs locally?"
print(f"Q: {question}")
print(f"A: {chain.invoke(question)}")
vectorstore.delete_collection()